In [10]:
import kan
from spin_systems import SpinSystem, spin_system, heisenberg, zero_sector_basis, no_symmetries_basis
from spin_lattices import SquareLattice, KagomeLattice
import torch
from typing import Callable
import numpy.typing as npt
import numpy as np

In [2]:
lattice = SquareLattice(4, 4)
J2 = 0
system = spin_system(heisenberg(lattice, J2=J2), no_symmetries_basis())

In [46]:
from fourier_supervised_cleanroom import sign_signal


def create_dataset(
    system: SpinSystem, signal: Callable[[npt.NDArray[np.uint64]], npt.NDArray],
    n_train: int, n_test: int, device: torch.device
) -> dict[str, torch.Tensor]:
    inputs = np.random.choice(system.basis.states, n_train + n_test, replace=False)
    train_input = inputs[:n_train]
    test_input = inputs[n_train:]
    train_output = signal(train_input)
    test_output = signal(test_input)
    return {
        "train_input": system.lattice.unpack_configurations(torch.tensor(train_input.astype(np.int64)).to(device)),
        "train_label": torch.tensor(train_output).to(device),
        "test_input": system.lattice.unpack_configurations(torch.tensor(test_input.astype(np.int64)).to(device)),
        "test_label": torch.tensor(test_output).to(device),
    }

In [47]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [65]:
dataset = create_dataset(
    system,
    #lambda x: sign_signal(system, tol=1e-7, apply_symmetries=False)(x).astype(np.int64),
    lambda x: system.lattice.unpack_configurations(x)[:, 0].astype(np.int64),
    int(system.basis.states.shape[0] * 0.7),
    int(system.basis.states.shape[0] * 0.3),
    device=device,
)

In [66]:
dataset

{'train_input': tensor([[0, 0, 1,  ..., 0, 1, 0],
         [0, 1, 0,  ..., 1, 1, 1],
         [0, 1, 0,  ..., 0, 1, 1],
         ...,
         [1, 1, 0,  ..., 0, 1, 1],
         [1, 0, 1,  ..., 1, 0, 0],
         [1, 1, 0,  ..., 1, 1, 0]], device='cuda:0'),
 'train_label': tensor([0, 0, 0,  ..., 1, 1, 1], device='cuda:0'),
 'test_input': tensor([[1, 0, 0,  ..., 1, 0, 0],
         [0, 1, 0,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 0, 0],
         ...,
         [0, 0, 1,  ..., 0, 0, 1],
         [0, 0, 0,  ..., 0, 1, 1],
         [0, 1, 1,  ..., 0, 0, 1]], device='cuda:0'),
 'test_label': tensor([1, 0, 1,  ..., 0, 0, 0], device='cuda:0')}

In [70]:
model = kan.KAN(width=[system.number_spins, 1], grid=2, k=1, device=str(device))

In [71]:
model.train(dataset, opt="LBFGS", steps=2, )

train loss: 5.00e-01 | test loss: 5.00e-01 | reg: 8.70e+00 : 100%|████| 2/2 [00:20<00:00, 10.33s/it]


{'train_loss': [array(0.49999578), array(0.49999578)],
 'test_loss': [array(0.5000239), array(0.5000239)],
 'reg': [array(8.69857298), array(8.69857298)]}

In [76]:
model.auto_symbolic()

fixing (0,0,0) with exp, r2=1.0000000000000018
fixing (0,1,0) with exp, r2=1.0000000000000018


KeyboardInterrupt: 